# Modern Transformer Architecture - Homework

In this assignment, you'll implement key innovations in modern Transformers:
- **RMSNorm**: Simpler and faster normalization
- **RoPE**: Rotary positional embeddings for better length extrapolation
- **Simplified MHLA**: Efficient attention with reduced KV cache

**What you'll learn:**
- Why modern LLMs use these techniques
- How they improve efficiency and performance
- Trade-offs between different approaches

**Submission:** Submit this completed notebook and completed python files.

**Grading:**
- Part 1 (RMSNorm): 20 points
- Part 2 (RoPE): 30 points
- Part 3 (MHLA): 35 points
- Part 4 (Integration + Questions): 15 points
Total: 100 points

## Setup

See `runner.py`.

## Part 1: RMSNorm (20 points)

See `RMSNorm.py`.

See `runner.py`.

**Question 1.1 (5 points - Written):** Why doesn't RMSNorm need to compute the mean? In what way is "controlling scale" sufficient for gradient stability?

**Why RMSNorm doesn't need to compute the mean**: RMSNorm doesn't need to compute the mean because the input is already approximately centered due to how residual streams function, so computing the mean would add little benefit.

**In what way "controlling scale" is sufficient for gradient stability**: Gradient instability is fundamentally a magnitude problem — gradients explode or vanish based on how large or small activations are, not where they're centered. By keeping activation magnitudes consistent across layers, RMSNorm ensures gradients stay in a stable range as they multiply back through the network.

## Part 2: Rotary Positional Embeddings (30 points)

See `RoPE.py`.

See `runner.py`.

**Question 2.1 (5 points - Written):** Why does RoPE generalize better to longer sequences than learned absolute positional embeddings?

**Why RoPE generalizes better to longer sequences than learned absolute positional embeddings**: Learned absolute position embeddings assign a unique learned vector to each position index. If the model never saw position 5000 during training, it has no embedding for it. RoPE encodes position through rotation angles derived from a deterministic formula, so any position can be computed — it never needs to have been "seen." Also, because the dot product of rotated Q and K depends only on the relative distance between positions, the model naturally handles relative positions it saw during training even at new absolute positions.

# Part 3: Simplified Multi-Head Latent Attention (35 points)

See `MHLA.py`.

See `runner.py`.

**Question 3.1 (5 points - Written):** The compression ratio is constant regardless of sequence
length. Why? What does this tell you about where the memory savings come from?

**Why the compression ratio is constant regardless of sequence length**: The compression happens per-token: each token's K and V information is compressed from 2 × d_model down to d_latent regardless of how many tokens exist. The resulting ratio is d_latent / (2 × d_model). 

**What this tells about where memory savings come from**: The momory savings comes from the per-token cache size instead of the sequence-length-dependent.

**Question 3.2 (5 points - Written):** MHLA compresses K and V significantly. What information
might be lost? In what scenarios might this hurt model quality?

**What information might be lost**: Potential info that could be lost are fine-grained token-specific details that don't survive the low-rank projection. 

**In what scenarios might this hurt model quality**: This could hurt tasks requiring precise retrieval of specific token values, like copying exact numbers or names, or tasks where many semantically distinct tokens need to be distinguished. 

## Part 4: Putting It All Together (15 points)

See `Transformer.py`.

See `runner.py`.

## Final Questions

### Final Questions

**Question 4.1 (3 points - Written):** Imagine you're deploying a chatbot that needs to handle
conversations of 50,000 tokens. Would you use standard attention or MHLA?
Why? What would be the memory savings?

**Which of standard attention or MHLA you would use**: I would use MHLA

**Why**:  Standard attention caches 2 × n_heads × d_head values per token; MHLA caches d_latent per token. With typical values, the cache per token can shrink drastically.

**What the memory savings would be**: The total KV cache goes from ~1.6GB to ~100MB (in float16) across 50,000 tokens.

**Question 4.2 (3 points - Written):** What's the main trade-off when using a smaller latent dimension
$d_{\text{latent}}$ in MHLA? How would you choose this hyperparameter?

**What the main trade-off when using a smaller latent dimension in MHLA is**: More memory savings but more information loss.

**How you would choose the latent dimension**: By starting with a value like d_model / 4 or d_model / 8 and evaluating downstream task quality, looking for the smallest value where performance doesn't degrade meaningfully.

**Question 4.3 (4 points - Written):** We used Pre-LN (normalize before the block) instead of Post-LN (normalize after the residual). Why is Pre-LN better for deep networks? Hint: think about gradient flow.

**Why Pre-LN is better for deep networks**: In Post-LN, gradients must pass through the normalization layer after the residual addition on the backward pass, which can shrink them in early layers of a deep network. In Pre-LN on the other hand, gradients flow directly through the addition back to earlier layers without passing through a normalization, keeping gradient magnitudes stable throughout training depth.